In [18]:
import json
import pandas as pd
import os

data = []
base_path = os.path.dirname(os.getcwd())
log_dir = os.path.join(base_path, "log")

file_path = os.path.join(log_dir, "cowrie_all.json")
df = pd.read_json(file_path, lines=True)

print(df.tail(50))

                          eventid           src_ip  src_port         dst_ip  \
7079       cowrie.session.connect  172.236.119.165   61082.0  172.31.26.155   
7080        cowrie.client.version  172.236.119.165       NaN            NaN   
7081        cowrie.session.closed  172.236.119.165       NaN            NaN   
7082       cowrie.session.connect    70.166.167.49   44940.0  172.31.26.155   
7083        cowrie.session.closed    70.166.167.49       NaN            NaN   
7084       cowrie.session.connect      49.207.9.39   53058.0  172.31.26.155   
7085        cowrie.client.version      49.207.9.39       NaN            NaN   
7086            cowrie.client.kex      49.207.9.39       NaN            NaN   
7087         cowrie.login.success      49.207.9.39       NaN            NaN   
7088  cowrie.direct-tcpip.request      49.207.9.39   21298.0   77.88.21.158   
7089        cowrie.session.closed      49.207.9.39       NaN            NaN   
7090       cowrie.session.connect  183.247.171.186  

In [19]:
sessions = []
print(df.columns)
grouped = df.groupby("session") #Agrupas el df por sesión
for session_id, group in grouped:
    #Diccionario de esta sesion
    session_data = {}

    #Guarda ID de la sesión
    session_data["session"] = session_id

    #LO BÁSICO
    #Protocolo
    session_data["protocol"] = group["protocol"].dropna().iloc[0] if not group["protocol"].dropna().empty else None

    # IP atacante
    session_data["src_ip"] = group["src_ip"].dropna().iloc[0] if not group["src_ip"].dropna().empty else None

    # Puerto origen 
    session_data["src_port"] = group["src_port"].dropna().iloc[0] if not group["src_port"].dropna().empty else None

    session_data["username"] = group["username"].dropna().iloc[0] if not group["username"].dropna().empty else None
    session_data["password"] = group["password"].dropna().iloc[0] if not group["password"].dropna().empty else None

    #SESIÓN
    # Duración segun evento cierre
    duration = group[group["eventid"] == "cowrie.session.closed"]["duration"]
    session_data["duration"] = float(duration.iloc[0]) if not duration.empty else 0
    
    login_success = int((group["eventid"] == "cowrie.login.success").any())
    login_failed = (group["eventid"] == "cowrie.login.failed").sum()

    session_data["login_success"] = login_success
    session_data["login_attempts"] = login_failed
    
    #Calcular la proporción de intentos de login fallidos respecto al total
    session_data["auth_failure_ratio"] = (
    session_data["login_attempts"] /
    (session_data["login_attempts"] + session_data["login_success"] + 1)
    )

    #COMANDOS
    # Nº comandos ejecutados por el atacante 
    session_data["num_commands"] = (group["eventid"] == "cowrie.command.input").sum()

    # Descarga de archivos en la sesión (1 = sí, 0 = no)
    session_data["file_download"] = int((group["eventid"] == "cowrie.session.file_download").any())

    #Comandos atacante
    commands = group[group["eventid"] == "cowrie.command.input"]["input"].dropna().tolist()
    session_data["commands"] = commands

    if commands:
        session_data["recon"] = int(any(x in cmd.lower() for cmd in commands for x in ["uname","whoami","id","ifconfig","pwd"]))
        session_data["download"] = int(any(x in cmd.lower() for cmd in commands for x in ["wget","curl","tftp"]))
        session_data["chmod"] = int(any("chmod" in cmd.lower() for cmd in commands))
        session_data["execution"] = int(any(x in cmd.lower() for cmd in commands for x in ["./","sh ","bash "]))
        session_data["persistence"] = int(any(x in cmd.lower() for cmd in commands for x in [".ssh","authorized_keys"]))

        session_data["unique_commands"] = len(set(commands))
        session_data["avg_command_length"] = sum(map(len, commands)) / len(commands)

    else:
        session_data["recon"] = 0
        session_data["download"] = 0
        session_data["chmod"] = 0
        session_data["execution"] = 0
        session_data["persistence"] = 0

        session_data["unique_commands"] = 0
        session_data["avg_command_length"] = 0
        
    #VELOCIDAD
    # Velocidad del ataque, alto->bot, bajo -> humano
    session_data["commands_per_second"] = (
        session_data["num_commands"] / session_data["duration"]
        if session_data["duration"] > 0 else 0
    )

    #COMPORTAMIENTO DE AUTENTICACIÓN SIN COMANDOS
    #Intensidad intentando adivinar
    session_data["credential_guessing_intensity"] = int(login_failed >= 5)

    #Probable que sea bot
    session_data["bot_likelihood_auth"] = int(
        login_failed >= 3 and session_data["duration"] < 5
    )

    #Puntuación de serveridad de autenticación
    session_data["auth_severity_score"] = (
        login_failed * 2 +
        login_success * 5 +
        session_data["duration"] * 0.1
    )

    # Tipo de sesión sin comandos
    if session_data["num_commands"] == 0:
        #Muchos intentos fallidos → posible ataque de fuerza bruta.
        if login_failed >= 5:
            session_data["session_stage"] = "bruteforce"
        #Login exitoso, pero no se ejecutaron comandos → alguien entró pero no hizo nada.
        elif login_success:
            session_data["session_stage"] = "login_success_no_shell"
        #Sesión larga sin comandos → posible escaneo lento
        elif session_data["duration"] > 60:
            session_data["session_stage"] = "slow_probe"
        #Actividad muy básica → probablemente un escaneo automático.
        else:
            session_data["session_stage"] = "scan"
     #Interracción con el sistema   
    else:
        session_data["session_stage"] = "interactive"

    #Clasificación ataques
    attack_types = []

    if session_data["recon"]:
        attack_types.append("reconnaissance")

    if session_data["download"]:
        attack_types.append("malware_download")

    if session_data["execution"]:
        attack_types.append("execution")

    if session_data["persistence"]:
        attack_types.append("persistence")

    if session_data["chmod"]:
        attack_types.append("privilege_change")

    if login_failed > 5:
        attack_types.append("bruteforce")

    if session_data["file_download"]:
        attack_types.append("file_activity")

    if session_data["commands_per_second"] > 2:
        attack_types.append("bot")

    if session_data["unique_commands"] > 5:
        attack_types.append("interactive_attack")

    #Si no hay comandos, clasificación ataques
    if session_data["num_commands"] == 0:

        if login_failed >= 5:
            attack_types.append("auth_bruteforce")

        elif login_success:
            attack_types.append("auth_success_no_action")

        elif session_data["auth_severity_score"] > 10:
            attack_types.append("high_auth_pressure")

        else:
            attack_types.append("scan_or_noise")

    session_data["attack_type"] = attack_types
    #Agregar el diccionario
    sessions.append(session_data)
    
# Crea dataframe a partir de todos los diccionarios
dataset = pd.DataFrame(sessions)
print(dataset.head(50))
dataset.to_csv("../log/dataset.csv", index=False)

Index(['eventid', 'src_ip', 'src_port', 'dst_ip', 'dst_port', 'session',
       'protocol', 'message', 'sensor', 'uuid', 'timestamp', 'version',
       'hassh', 'hasshAlgorithms', 'kexAlgs', 'keyAlgs', 'encCS', 'macCS',
       'compCS', 'langCS', 'duration', 'username', 'password', 'arch', 'input',
       'ttylog', 'size', 'shasum', 'duplicate', 'outfile', 'destfile', 'width',
       'height', 'name', 'value', 'command', 'option_name', 'option_byte'],
      dtype='object')
         session protocol           src_ip  src_port        username  \
0   0033683edcd9      ssh      65.20.175.6   35567.0          centos   
1   00ae7ffbb20a      ssh    117.216.33.31   40544.0           admin   
2   00dd83768373      ssh  220.246.194.124   47791.0           admin   
3   010705ba79a9      ssh   35.130.111.146   35155.0           admin   
4   0175c5fd6f01      ssh     183.82.97.80   39308.0          config   
5   01efdd37d845      ssh      177.174.0.3   56125.0            User   
6   020c2cb3291e  

In [4]:
# Cuenta de cuantas veces nos ha atacado cada IP.
group_by_ip = dataset["src_ip"].value_counts()
print(group_by_ip)


src_ip
64.227.149.124     12
101.206.107.245    10
127.0.0.1           5
193.146.227.231     5
47.122.77.248       2
209.38.83.142       2
120.241.79.66       2
46.101.72.225       2
220.203.237.173     2
106.12.75.58        2
52.180.145.88       2
47.238.152.243      2
20.64.104.5         2
172.202.118.21      2
49.7.210.65         2
20.121.123.108      2
204.76.203.73       2
35.203.211.8        2
35.203.211.155      2
135.237.123.203     2
162.216.149.133     2
162.216.149.10      1
204.76.203.226      1
120.86.119.165      1
35.203.210.80       1
45.33.12.122        1
204.76.203.224      1
106.12.161.169      1
43.224.126.107      1
180.76.172.156      1
101.35.146.20       1
31.56.209.39        1
147.185.132.37      1
35.203.210.142      1
203.19.35.152       1
183.56.235.140      1
103.64.129.98       1
160.119.76.24       1
35.203.211.34       1
162.216.150.109     1
Name: count, dtype: int64


In [ ]:
import geoip2.database

# Localizar IP por países
# Uso de la base de datos GeoLite2-City
# Página oficial de la base de datos https://www.maxmind.com/
reader = geoip2.database.Reader('../db/GeoLite2-City.mmdb')

paises_ataques = {}
for ip, numero_de_ataques in  group_by_ip.items():
    if ip == "127.0.0.1":
        continue
    response = reader.city(ip)
    
    if response.country.name not in paises_ataques.keys():
        paises_ataques[response.country.name] = numero_de_ataques
    else:
        paises_ataques[response.country.name] += numero_de_ataques

reader.close()

for pais, num_ataques in paises_ataques.items():
    print("Numero de ataques de:",pais , ": " , num_ataques)

ModuleNotFoundError: No module named 'geoip2'

In [ ]:
# Listamos todos los comandos ejecutados
dataset_filtrado = dataset[dataset["commands"].str.len() > 0]

set_comandos = set([x for xs in dataset_filtrado["commands"].tolist() for x in xs])
for idx, comando in enumerate(set_comandos):
    print(idx, " ", comando)
    

In [ ]:
# Agrupamos las conexiones en base a las duraciones que han tenido
# Si la duración ha durado menos de 20 segundos, lo consideramos corto
# Si la duración ha durado entre 20 y 80 segundos, lo consideramos medio
# Si la duración ha durado más de 80 segundos, lo consideramos largo

bajo = 0
med = 0
alto = 0
for duracion in dataset["duration"]:
    if duracion < 20:
        bajo += 1
    if 20 <= duracion < 80:
        med += 1
    if duracion >= 80:
        alto += 1

print("Conexiones de baja duración: ", bajo)
print("Conexiones de media duración: ", med)
print("Conexiones de alta duración: ", alto)

In [ ]:
print(dataset["protocol"].value_counts())

In [ ]:
# El comando "chmod" se usa para cambiar permisos en archivos y ficheros.
# Mediante este comando el intruso puede atribuirse permisos que en realidad
# no tiene.
# Contamos la cantidad de sesiones en la que se han cambiado los permisos.
si = 0
no = 0
for i in dataset["chmod"]:
    if i != 0:
        si +=1
    else:
        no += 1

print("Numero de sesiones donde han cambiado permisos", si)
print("Numero de sesiones donde no han cambiado permisos", no)

print(dataset["file_download"].value_counts())

In [ ]:
print(dataset["username"].value_counts())
print(dataset["password"].value_counts())

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

data_scaled = scaler.fit(dataset[["duration","login_attempts","num_commands","file_download","recon","download","chmod","execution","persistence","commands_per_second"]])
X = np.array(dataset[["duration","login_attempts","num_commands","file_download","recon","download","chmod","execution","persistence","commands_per_second"]])
pca = PCA(n_components=4)
pca_results = pca.fit_transform(X)
fig = plt.figure()
ax = plt.axes(projection='3d')
ax.scatter(pca_results[:,0],pca_results[:,1],pca_results[:,2], c=pca_results[:,3])

plt.show()
